In [ ]:
!pip install -q -U transformers accelerate
!pip install -q qwen-vl-utils jiwer pillow

In [ ]:
import os
import glob
import torch
from PIL import Image
from jiwer import cer
from transformers import AutoProcessor, AutoModelForCausalLM
from qwen_vl_utils import process_vision_info

# --- CẤU HÌNH ĐƯỜNG DẪN ---
DATA_DIR = "/kaggle/input/datasets/anhlandibi/vietnamese-original/vietnamese" 
IMAGE_DIR = os.path.join(DATA_DIR, "test_image")
LABEL_DIR = os.path.join(DATA_DIR, "labels")

def parse_ground_truth(label_path):
    """Đọc file label VinText, bỏ qua '###' và gom text"""
    valid_texts = []
    if not os.path.exists(label_path):
        return ""
    
    with open(label_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            
            parts = line.split(',')
            if len(parts) >= 9:
                text = ','.join(parts[8:]).strip()
                if text != '###':
                    valid_texts.append(text)
                    
    return " ".join(valid_texts)

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
import torch

model_id = "Qwen/Qwen3-VL-8B-Instruct"

print(f"Đang tải model {model_id}...")

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id, 
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

processor = AutoProcessor.from_pretrained(
    model_id, 
    trust_remote_code=True
)

print("Tải model thành công!")

In [ ]:
def qwen_ocr_predict(image_path):
    # Prompt yêu cầu gắt gao để model không sinh ra text thừa
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": "Hãy trích xuất toàn bộ văn bản tiếng Việt xuất hiện trong hình ảnh này. Chỉ in ra kết quả văn bản, nối với nhau bằng khoảng trắng, tuyệt đối không giải thích hay chat thêm."},
            ],
        }
    ]

    # Chuẩn bị input text
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    # Xử lý ảnh bằng hàm chuyên dụng của Qwen
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    # Sinh text
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=256)
        
    # Lọc bỏ phần prompt đầu vào khỏi output
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    
    return output_text[0].strip()

In [ ]:
import re 

image_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")))[:100]

results = []
total_cer = 0.0
valid_samples = 0

print(f"Bắt đầu đánh giá trên {len(image_files)} ảnh...")

for idx, img_path in enumerate(image_files):
    filename = os.path.basename(img_path)
    
    match = re.search(r'\d+', filename)
    if not match:
        print(f"[{idx+1}] Bỏ qua {filename} (Không tìm thấy số trong tên ảnh)")
        continue
        
    img_num = str(int(match.group())) 
    
    label_filename = f"gt_{img_num}.txt" 
    label_path = os.path.join(LABEL_DIR, label_filename)
    
    if not os.path.exists(label_path):
        label_path = os.path.join(LABEL_DIR, f"gt_{match.group()}.txt")
        
    ground_truth = parse_ground_truth(label_path)
    
    if not ground_truth:
        print(f"[{idx+1}] Bỏ qua {filename} (Không tìm thấy file label {label_filename} hoặc label trống)")
        continue
        
    prediction = qwen_ocr_predict(img_path)
    
    try:
        error_rate = cer(ground_truth.lower(), prediction.lower())
    except ValueError:
        error_rate = 1.0 
        
    total_cer += error_rate
    valid_samples += 1
    
    results.append({
        "image": filename,
        "gt": ground_truth,
        "pred": prediction,
        "cer": error_rate
    })
    
    print(f"[{idx+1}/{len(image_files)}] {filename} | CER: {error_rate:.4f}")
    print(f"  GT  : {ground_truth}")
    print(f"  PRED: {prediction}")
    print("-" * 50)

if valid_samples > 0:
    mean_cer = total_cer / valid_samples
    print(f"\n🚀 ĐÁNH GIÁ HOÀN TẤT!")
    print(f"Tổng số ảnh hợp lệ: {valid_samples}")
    print(f"Mean CER (Character Error Rate): {mean_cer:.4f}")
else:
    print("\nVẫn không tìm thấy ảnh hợp lệ. Vui lòng kiểm tra lại đường dẫn LABEL_DIR.")